# Qwen3-8B domain LoRA finetune (Unsloth)

Template on `main`. Each `domain/<name>` branch ships its own copy with `DOMAIN`
already set — see `docs/ADDING_A_DOMAIN.md`.

**Before running: Runtime → Change runtime type → GPU** (T4 works; A100 is faster).

The order below is the pipeline, and it matters:

1. prep data (dedup → decontaminate → hold out an unseen slice)
2. eval the **base** model → `baseline.json`
3. train the LoRA adapter
4. eval the **finetuned** model with the identical harness → `finetuned.json`
5. diff them into `report.md`

Which evals run is declared per domain in `domains/<name>/data_config.yaml` under `evals:`.

In [ ]:
DOMAIN = "finance"

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> GPU, then rerun this cell."
)
print(torch.cuda.get_device_name(0))

In [ ]:
# Unsloth's recommended install tracks Colab's CUDA/torch build and changes over
# time — if this cell fails, check https://github.com/unslothai/unsloth for the
# current one.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q evalplus datasets datasketch huggingface_hub pyyaml

In [ ]:
import os
import sys

REPO_URL = "https://github.com/rchhabra13/llm-finetune-lab.git"

if not os.path.exists("llm-finetune-lab"):
    !git clone -b domain/$DOMAIN $REPO_URL
%cd llm-finetune-lab

sys.path.insert(0, os.getcwd())  # so `import src...` resolves

## 1. Data prep

Download → exact + near-duplicate dedup → decontaminate against the benchmarks this
domain is scored on → hold out an unseen slice. No GPU needed; the same commands run
locally. Stats land in `domains/<DOMAIN>/results/`.

In [ ]:
!python -m src.data.download --config domains/$DOMAIN/data_config.yaml
!python -m src.data.dedup --config domains/$DOMAIN/data_config.yaml
!python -m src.data.decontaminate --config domains/$DOMAIN/data_config.yaml

## 2. Load the base model

In [ ]:
import yaml
from unsloth import FastLanguageModel

base_cfg = yaml.safe_load(open("configs/base.yaml"))
data_cfg = yaml.safe_load(open(f"domains/{DOMAIN}/data_config.yaml"))
lora_cfg, training_cfg = base_cfg["lora"], base_cfg["training"]
base_model = base_cfg["base_model"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=base_cfg["max_seq_length"],
    load_in_4bit=base_cfg["load_in_4bit"],
)

## 3. Baseline eval — base model, before any training

Runs as a subprocess against a clean base model, so nothing from the training cells
can leak into it. This is what "finetuned" is measured against; skip it and the final
numbers mean nothing.

**Slowest cell in the notebook** — it generates an answer for every eval problem.

In [ ]:
!python -m src.eval.run_all \
    --config domains/$DOMAIN/data_config.yaml \
    --model-path {base_model} \
    --out domains/$DOMAIN/results/baseline.json

## 4. Format the training data

Prompt shape comes from the `chat:` block in the domain's `data_config.yaml`, built by
the same `build_messages` the held-out eval uses — so training and eval can't drift
apart.

Read the printed example before training. If it looks wrong, fix `chat:` rather than
burning a GPU hour on a bad prompt shape.

In [ ]:
import json
from datasets import Dataset
from src.data.chat_format import build_messages

chat_cfg = data_cfg["chat"]
records = [json.loads(l) for l in open(f"domains/{DOMAIN}/data/processed/train.jsonl")]

def to_text(record):
    return {"text": tokenizer.apply_chat_template(build_messages(record, chat_cfg), tokenize=False)}

dataset = Dataset.from_list(records).map(to_text, remove_columns=list(records[0].keys()))
print(dataset)
print("\n--- one formatted example ---\n")
print(dataset[0]["text"][:2000])

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["alpha"],
    lora_dropout=lora_cfg["dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    use_gradient_checkpointing="unsloth",
    random_state=training_cfg["seed"],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=base_cfg["max_seq_length"],
    args=SFTConfig(
        per_device_train_batch_size=training_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=training_cfg["gradient_accumulation_steps"],
        num_train_epochs=training_cfg["num_train_epochs"],
        learning_rate=training_cfg["learning_rate"],
        lr_scheduler_type=training_cfg["lr_scheduler_type"],
        warmup_ratio=training_cfg["warmup_ratio"],
        weight_decay=training_cfg["weight_decay"],
        optim=training_cfg["optim"],
        seed=training_cfg["seed"],
        output_dir=f"domains/{DOMAIN}/adapters/checkpoints",
        logging_steps=10,
        report_to="none",
        save_strategy="no",
    ),
)

trainer_stats = trainer.train()

In [ ]:
adapter_dir = f"domains/{DOMAIN}/adapters/final"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

with open(f"domains/{DOMAIN}/results/train_stats.json", "w") as f:
    json.dump(
        {
            "train_runtime_sec": trainer_stats.metrics.get("train_runtime"),
            "train_loss": trainer_stats.metrics.get("train_loss"),
            "train_examples": len(dataset),
            "base_model": base_model,
            "lora_r": lora_cfg["r"],
            "epochs": training_cfg["num_train_epochs"],
            "gpu": torch.cuda.get_device_name(0),
        },
        f,
        indent=2,
    )
print(trainer_stats.metrics)

## 6. Finetuned eval — identical harness, now with the adapter

Same script, same seed, same benchmarks as step 3. The only difference is
`--adapter-path`. That is what makes the diff meaningful.

In [ ]:
!python -m src.eval.run_all \
    --config domains/$DOMAIN/data_config.yaml \
    --model-path {base_model} \
    --adapter-path {adapter_dir} \
    --out domains/$DOMAIN/results/finetuned.json

## 7. Report

In [ ]:
!python -m src.eval.report --domain $DOMAIN

## 8. Try it yourself (optional)

The numbers above are the result; this just shows what the adapter actually produces
on a held-out example it never trained on.

In [ ]:
FastLanguageModel.for_inference(model)

heldout = [json.loads(l) for l in open(f"domains/{DOMAIN}/data/processed/heldout.jsonl")]
messages = build_messages(heldout[0], chat_cfg)[:-1]  # drop the reference answer

inputs = tokenizer(
    tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False),
    return_tensors="pt",
).to(model.device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=False)

print("PROMPT:\n", messages[-1]["content"][:800])
print("\nMODEL:\n", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print("\nREFERENCE:\n", build_messages(heldout[0], chat_cfg)[-1]["content"][:800])

## 9. Commit results back

Adapter weights are gitignored — the results and stats are what belong in the repo.

```bash
git add domains/$DOMAIN/results
git commit -m "$DOMAIN domain: baseline vs finetuned results"
git push
```

Then add the row to `docs/RESULTS.md` and update the status table in the root `README.md`.